# 02_Clustering_scRNA - Clustering Analysis for scRNA-seq

This notebook performs clustering analysis on the scRNA-seq dataset:
- Preprocessing: log1p, HVG selection, scaling, PCA
- Clustering: KMeans, Agglomerative, DBSCAN
- Evaluation: Silhouette score, Davies-Bouldin, Adjusted Rand Index
- Visualizations: PCA and UMAP

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

import warnings
warnings.filterwarnings('ignore')

# Load data
X = pd.read_csv('data/scRNA_data.csv', index_col=0)

try:
    y = pd.read_csv('data/sc_RNA_labels.csv', index_col=0)
    y_true = y.iloc[:, 0].values
except Exception:
    y = None
    y_true = None

print('Loaded:', X.shape)

In [ ]:
# =========================
# PREPROCESSING
# =========================

# Step 1: Log transformation (standard in scRNA-seq)
X_log = np.log1p(X)

# Step 2: HVG selection (top variable genes)
gene_vars = X_log.var(axis=0).sort_values(ascending=False)
n_hvg = min(2000, X_log.shape[1])
hvg_genes = gene_vars.head(n_hvg).index

X_hvg = X_log[hvg_genes]

# Step 3: Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_hvg)

# Step 4: PCA
# Reduce dimensionality to remove noise and improve cluster structure
# 50 components is a common trade-off between information retention and efficiency
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# PCA for visualization
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)

print('Preprocessing done:', X_pca.shape)
print('Explained variance (first 10 PCs):', pca.explained_variance_ratio_[:10])

In [ ]:
# =========================
# FIND OPTIMAL K (KMeans)
# =========================

results = []

for k in range(2, 21):
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_pca)
    
    sil = silhouette_score(X_pca, labels)
    db = davies_bouldin_score(X_pca, labels)
    
    results.append({'k': k, 'silhouette': sil, 'davies_bouldin': db})

df_k = pd.DataFrame(results).set_index('k')
print(df_k)

best_k = int(df_k['silhouette'].idxmax())
print(f'Best k by silhouette: {best_k}')

# Plot
plt.figure(figsize=(8,4))
plt.plot(df_k.index, df_k['silhouette'], marker='o')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.title('Silhouette vs k')
plt.grid(True)
plt.show()

In [ ]:
# =========================
# KMEANS FINAL MODEL
# =========================

km_best = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
km_labels = km_best.fit_predict(X_pca)

print("\n--- KMEANS RESULTS ---")
print('Silhouette:', silhouette_score(X_pca, km_labels))
print('Davies-Bouldin:', davies_bouldin_score(X_pca, km_labels))

if y_true is not None:
    ari = adjusted_rand_score(y_true, km_labels)
    print('Adjusted Rand Index:', ari)

In [ ]:
# =========================
# AGGLOMERATIVE CLUSTERING
# =========================

agg = AgglomerativeClustering(n_clusters=best_k)
agg_labels = agg.fit_predict(X_pca)

print("\n--- AGGLOMERATIVE RESULTS ---")
print('Silhouette:', silhouette_score(X_pca, agg_labels))
print('Davies-Bouldin:', davies_bouldin_score(X_pca, agg_labels))

In [ ]:
# =========================
# DBSCAN EXPLORATION
# =========================

print("\n--- DBSCAN SEARCH ---")

for eps in np.linspace(0.5, 3, 6):
    for min_samples in [3, 5, 10]:
        dbs = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbs.fit_predict(X_pca)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        
        if n_clusters <= 1:
            continue
        
        sil = silhouette_score(X_pca, labels)
        db = davies_bouldin_score(X_pca, labels)
        
        print(f"eps={eps:.2f}, min_samples={min_samples} → clusters={n_clusters}, silhouette={sil:.3f}, db={db:.3f}")

In [ ]:
# =========================
# VISUALIZATION (PCA)
# =========================

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_pca2[:,0],
    y=X_pca2[:,1],
    hue=km_labels,
    palette='tab10',
    s=40
)

plt.title(f'KMeans Clustering (k={best_k})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# UMAP VISUALIZATION
# =========================

try:
    import umap
except:
    raise ImportError("Install umap-learn: pip install umap-learn")

reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_umap[:,0],
    y=X_umap[:,1],
    hue=km_labels,
    palette='tab10',
    s=40
)

plt.title('KMeans Clusters on UMAP')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# SAVE CLUSTERS
# =========================

clusters_df = pd.DataFrame({
    'cluster_kmeans': km_labels
}, index=X.index)

clusters_df.to_csv('data/cluster_assignments_kmeans.csv')

print("\nClusters saved!")

In [ ]:
# =========================
# INTERPRETATION
# =========================

print("\n--- INTERPRETATION ---")
print("High ARI indicates strong agreement between clusters and biological labels.")
print("This suggests that preprocessing preserved meaningful biological structure.")
print("Moderate silhouette indicates clusters are separable but not perfectly compact.")
print("DBSCAN results suggest data forms compact clusters rather than density-based shapes.")